# Amazon ML Challenge 2026 — Adaptive Entity Resolution Pipeline

### Key Components:
1. **Task 1**: Exact Macro $F_{0.5}$ metric per Source 1 entity (1.0 for true singletons, 0.0 for false merges).
2. **Task 2**: Explicit Measurement of Blocking Recall Ceiling & candidate reduction efficiency.
3. **Task 3**: Hard Negative Mining from blocking near-misses & Monotonic LightGBM.
4. **Parallel Representations**: Unicode NFKD Normalized, Delimited Tokens, Compact Signatures, Numeric Profiles.
5. **Dynamic Corpus IDF**: Data-driven stopword downweighting across all countries (US, India, France).

In [ ]:
# 1. Install & import high-performance dependencies
!pip install -q rapidfuzz polars lightgbm gdown

import os
import sys
import time
import math
import json
import re
import pickle
import unicodedata
import zipfile
from collections import Counter, defaultdict, namedtuple
from typing import Dict, List, Optional, Set, Tuple, Union, Any

import numpy as np
import polars as pl
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler

print('All libraries imported successfully!')

In [ ]:
# 2. Dataset Auto-Discovery & Download
def setup_dataset():
    train_dir = None
    candidates = [
        '/kaggle/input',
        'dataset/student_resource/dataset/train',
        'dataset/train',
        '/tmp/dataset/student_resource/dataset/train',
        'student_resource/dataset/train'
    ]
    for cand in candidates:
        if os.path.exists(cand):
            for root, _, files in os.walk(cand):
                if 'train_ground_truth.tsv' in files and 'train_source1.tsv' in files:
                    print(f'Found existing dataset at: {root}')
                    return root

    print('Dataset not found locally. Downloading from Google Drive...')
    import gdown
    file_id = '1qgSaaxRLXfXh7nmTEQ30jn8A2p3TyRgM'
    url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(url, 'dataset.zip', quiet=False)
    
    print('Extracting dataset.zip...')
    with zipfile.ZipFile('dataset.zip', 'r') as zf:
        zf.extractall('dataset')
    
    for root, _, files in os.walk('dataset'):
        if 'train_ground_truth.tsv' in files and 'train_source1.tsv' in files:
            print(f'Extracted dataset found at: {root}')
            return root
            
    raise FileNotFoundError('Could not locate train_ground_truth.tsv after extraction.')

DATA_DIR = setup_dataset()
print(f'Active DATA_DIR: {DATA_DIR}')

In [ ]:
# 3. Four Parallel Representations & Text Normalization
RE_NON_ALPHANUM = re.compile(r'[^a-z0-9\s]+')
RE_DIGITS = re.compile(r'\b\d+\b')

FEATURE_NAMES = [
    'name_jw',
    'name_jw_compact',
    'name_token_sort',
    'name_token_set',
    'name_char_3gram_jaccard',
    'name_char_4gram_jaccard',
    'name_idf_weighted_jaccard',
    'name_exact_compact',
    'addr_token_sort',
    'addr_token_set',
    'addr_char_3gram_jaccard',
    'addr_number_jaccard',
    'addr_number_conflict',
    'addr_number_match_count',
    'addr_is_null',
    'addr_both_null',
    'is_source2'
]

ParsedRecord = namedtuple(
    'ParsedRecord',
    [
        'name_norm',
        'name_sig',
        'name_words',
        'name_words_set',
        'name_3g',
        'name_4g',
        'addr_norm',
        'addr_words',
        'addr_nums',
        'addr_3g',
        'is_null',
        'first_street_word'
    ]
)

def normalize_text(text: Optional[str]) -> str:
    if text is None:
        return ''
    s = str(text)
    if not s or s == 'None' or s == 'nan':
        return ''
    s = s.replace('&', ' and ').replace('+', ' and ').replace('@', ' at ')
    s = ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c)).lower()
    return ' '.join(RE_NON_ALPHANUM.sub(' ', s).split())

def _extract_char_ngrams(s: str, n: int) -> Set[str]:
    if not s:
        return set()
    if len(s) < n:
        return {s}
    return {s[i:i + n] for i in range(len(s) - n + 1)}

def _extract_first_street_word(addr_words: List[str]) -> str:
    digits = [w for w in addr_words if w.isdigit()]
    if not digits:
        return ''
    first_digit = digits[0]
    try:
        idx = addr_words.index(first_digit)
        after = [w for w in addr_words[idx + 1:] if not w.isdigit() and len(w) >= 3]
        if after:
            return after[0]
        before = [w for w in addr_words[:idx] if not w.isdigit() and len(w) >= 3]
        if before:
            return before[-1]
    except ValueError:
        pass
    return ''

def parse_record(name: Optional[str], addr: Optional[str]) -> ParsedRecord:
    name_norm = normalize_text(name)
    if name_norm:
        name_words = name_norm.split()
        name_words_set = set(name_words)
        name_sig = ''.join(name_words)
        name_3g = _extract_char_ngrams(name_norm, 3)
        name_4g = _extract_char_ngrams(name_norm, 4)
    else:
        name_words = []
        name_words_set = set()
        name_sig = ''
        name_3g = set()
        name_4g = set()

    addr_norm = normalize_text(addr)
    if addr_norm:
        addr_words = addr_norm.split()
        addr_nums = {w for w in addr_words if w.isdigit()}
        addr_3g = _extract_char_ngrams(addr_norm, 3)
        is_null = 0.0
        street = _extract_first_street_word(addr_words)
    else:
        addr_words = []
        addr_nums = set()
        addr_3g = set()
        is_null = 1.0
        street = ''

    return ParsedRecord(
        name_norm=name_norm,
        name_sig=name_sig,
        name_words=name_words,
        name_words_set=name_words_set,
        name_3g=name_3g,
        name_4g=name_4g,
        addr_norm=addr_norm,
        addr_words=addr_words,
        addr_nums=addr_nums,
        addr_3g=addr_3g,
        is_null=is_null,
        first_street_word=street
    )

print('Parser defined successfully!')

In [ ]:
# 4. Dynamic Corpus IDF & Relative Feature Extraction
class CorpusIDF:
    def __init__(self, idf: Dict[str, float], stopwords: Set[str], df: Dict[str, int], default_idf: float, n_records: int):
        self.idf = idf
        self.stopwords = stopwords
        self.df = df
        self.default_idf = default_idf
        self.n_records = n_records

def compute_corpus_idf(parsed_records: List[ParsedRecord], top_n_stopwords: int = 150) -> CorpusIDF:
    df_counter = Counter()
    for r in parsed_records:
        df_counter.update(r.name_words_set)
    
    n_records = max(1, len(parsed_records))
    idf = {w: math.log((n_records + 1.0) / (cnt + 1.0)) for w, cnt in df_counter.items()}
    stopwords = {w for w, _ in df_counter.most_common(top_n_stopwords)}
    default_idf = math.log(n_records + 1.0)
    return CorpusIDF(idf=idf, stopwords=stopwords, df=dict(df_counter), default_idf=default_idf, n_records=n_records)

def compute_parsed_features(
    r1: ParsedRecord,
    r2: ParsedRecord,
    is_s2: bool,
    idf_dict: Dict[str, float],
    default_idf: float
) -> List[float]:
    jw_val = JaroWinkler.similarity(r1.name_norm, r2.name_norm)
    jw_comp = JaroWinkler.similarity(r1.name_sig, r2.name_sig)
    tok_sort = fuzz.token_sort_ratio(r1.name_norm, r2.name_norm) / 100.0
    tok_set = fuzz.token_set_ratio(r1.name_norm, r2.name_norm) / 100.0

    inter_3g = len(r1.name_3g & r2.name_3g)
    union_3g = len(r1.name_3g | r2.name_3g)
    c3_jacc = inter_3g / union_3g if union_3g > 0 else 0.0

    inter_4g = len(r1.name_4g & r2.name_4g)
    union_4g = len(r1.name_4g | r2.name_4g)
    c4_jacc = inter_4g / union_4g if union_4g > 0 else 0.0

    inter_words = r1.name_words_set & r2.name_words_set
    if inter_words:
        union_words = r1.name_words_set | r2.name_words_set
        num = sum(idf_dict.get(w, default_idf) for w in inter_words)
        den = sum(idf_dict.get(w, default_idf) for w in union_words)
        idf_jacc = num / den if den > 0 else 0.0
    else:
        idf_jacc = 0.0

    exact_comp = 1.0 if (r1.name_sig and r1.name_sig == r2.name_sig) else 0.0

    if r1.is_null > 0.5 or r2.is_null > 0.5:
        addr_sort = 0.0
        addr_set = 0.0
        a3_jacc = 0.0
        num_jacc = 0.0
        num_conf = 0.0
        num_match = 0.0
    else:
        addr_sort = fuzz.token_sort_ratio(r1.addr_norm, r2.addr_norm) / 100.0
        addr_set = fuzz.token_set_ratio(r1.addr_norm, r2.addr_norm) / 100.0
        a_inter = len(r1.addr_3g & r2.addr_3g)
        a_union = len(r1.addr_3g | r2.addr_3g)
        a3_jacc = a_inter / a_union if a_union > 0 else 0.0

        n1 = r1.addr_nums
        n2 = r2.addr_nums
        if n1 and n2:
            n_inter = len(n1 & n2)
            n_union = len(n1 | n2)
            num_jacc = n_inter / n_union
            num_conf = 1.0 if n_inter == 0 else 0.0
            num_match = float(n_inter)
        else:
            num_jacc = 0.0
            num_conf = 0.0
            num_match = 0.0

    addr_is_null = 1.0 if (r1.is_null > 0.5 or r2.is_null > 0.5) else 0.0
    addr_both_null = 1.0 if (r1.is_null > 0.5 and r2.is_null > 0.5) else 0.0
    is_s2_fl = 1.0 if is_s2 else 0.0

    return [
        jw_val,
        jw_comp,
        tok_sort,
        tok_set,
        c3_jacc,
        c4_jacc,
        idf_jacc,
        exact_comp,
        addr_sort,
        addr_set,
        a3_jacc,
        num_jacc,
        num_conf,
        num_match,
        addr_is_null,
        addr_both_null,
        is_s2_fl
    ]

print('Feature extraction pipeline ready!')

In [ ]:
# 5. Exact Macro F_0.5 Evaluation & Blocking Recall Ceiling Module (TASK 1 & TASK 2)
def compute_entity_f05(y_true: Set[str], y_pred: Set[str]) -> float:
    len_true = len(y_true)
    len_pred = len(y_pred)
    if len_true == 0:
        return 1.0 if len_pred == 0 else 0.0
    if len_pred == 0:
        return 0.0
    tp = len(y_true & y_pred)
    if tp == 0:
        return 0.0
    return (5.0 * tp) / (float(len_true) + 4.0 * float(len_pred))

def compute_macro_f05(ground_truth: Dict[str, Set[str]], predictions: Dict[str, Set[str]]) -> float:
    if not ground_truth:
        return 0.0
    total_score = sum(compute_entity_f05(y_true, predictions.get(sid, set())) for sid, y_true in ground_truth.items())
    return total_score / float(len(ground_truth))

def evaluate_blocking_recall(
    ground_truth: Dict[str, Set[str]],
    candidate_pairs: Dict[str, Set[str]],
    total_s2_pool_size: Optional[int] = None,
    total_s3_pool_size: Optional[int] = None
) -> Dict[str, Any]:
    total_true_matches = 0
    retained_matches = 0
    non_singleton_entities = 0
    full_coverage_entities = 0
    candidate_counts = []

    for sid, y_true in ground_truth.items():
        candidates = candidate_pairs.get(sid, set())
        candidate_counts.append(len(candidates))
        len_true = len(y_true)
        if len_true > 0:
            non_singleton_entities += 1
            total_true_matches += len_true
            captured = len(y_true & candidates)
            retained_matches += captured
            if captured == len_true:
                full_coverage_entities += 1

    recall_ceiling = (retained_matches / total_true_matches) if total_true_matches > 0 else 1.0
    entity_full_cov = (full_coverage_entities / non_singleton_entities) if non_singleton_entities > 0 else 1.0
    avg_cands = float(np.mean(candidate_counts)) if candidate_counts else 0.0

    report = {
        'blocking_recall_ceiling': recall_ceiling,
        'total_true_matches': total_true_matches,
        'retained_in_candidates': retained_matches,
        'missed_in_blocking': total_true_matches - retained_matches,
        'non_singleton_entities': non_singleton_entities,
        'entity_full_coverage_rate': entity_full_cov,
        'avg_candidates_per_s1': avg_cands,
        'max_candidates_per_s1': max(candidate_counts) if candidate_counts else 0
    }
    if total_s2_pool_size is not None and total_s3_pool_size is not None:
        total_possible = len(ground_truth) * (total_s2_pool_size + total_s3_pool_size)
        total_actual = sum(candidate_counts)
        report['reduction_ratio'] = 1.0 - (total_actual / total_possible) if total_possible > 0 else 1.0
    return report

def detailed_evaluation_report(ground_truth: Dict[str, Set[str]], predictions: Dict[str, Set[str]], candidate_pairs: Optional[Dict[str, Set[str]]] = None) -> Dict[str, Any]:
    singleton_scores = []
    non_singleton_scores = []
    total_tp = 0
    total_fp = 0
    total_fn = 0
    
    for sid, y_true in ground_truth.items():
        y_pred = predictions.get(sid, set())
        score = compute_entity_f05(y_true, y_pred)
        if len(y_true) == 0:
            singleton_scores.append(score)
        else:
            non_singleton_scores.append(score)
        tp = len(y_true & y_pred)
        total_tp += tp
        total_fp += len(y_pred - y_true)
        total_fn += len(y_true - y_pred)

    macro_f05 = compute_macro_f05(ground_truth, predictions)
    micro_prec = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    micro_rec = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0

    return {
        'macro_f05': macro_f05,
        'singleton_accuracy': float(np.mean(singleton_scores)) if singleton_scores else 1.0,
        'non_singleton_macro_f05': float(np.mean(non_singleton_scores)) if non_singleton_scores else 0.0,
        'global_micro_precision': micro_prec,
        'global_micro_recall': micro_rec,
        'singletons_evaluated': len(singleton_scores),
        'non_singletons_evaluated': len(non_singleton_scores)
    }

print('Exact Competition Evaluation & Validation Framework Loaded!')

In [ ]:
# 6. Adaptive Inverted Index for High-Efficiency Blocking
class AdaptiveInvertedIndex:
    def __init__(self, max_posting: int = 30, max_candidates: int = 15, stopwords: Optional[Set[str]] = None, df_dict: Optional[Dict[str, int]] = None):
        self.index = defaultdict(list)
        self.max_posting = max_posting
        self.max_candidates = max_candidates
        self.stopwords = stopwords if stopwords is not None else set()
        self.df = df_dict if df_dict is not None else {}

    def add_records(self, parsed_records: List[ParsedRecord], is_s2_flag: int):
        sw = self.stopwords
        df = self.df
        flag = 1 if is_s2_flag == 1 else 0
        for i, r in enumerate(parsed_records):
            sig = r.name_sig
            if len(sig) >= 5:
                k6 = f'p6_{sig[:6]}'
                lst6 = self.index[k6]
                if len(lst6) < self.max_posting:
                    lst6.append((flag, i))
                if len(sig) >= 8:
                    k8 = f'p8_{sig[:8]}'
                    lst8 = self.index[k8]
                    if len(lst8) < self.max_posting:
                        lst8.append((flag, i))
            for w in r.name_words_set:
                if len(w) >= 3 and (w not in sw):
                    if 2 <= df.get(w, 0) <= 300:
                        lst_tok = self.index[f't_{w}']
                        if len(lst_tok) < self.max_posting:
                            lst_tok.append((flag, i))
            if (r.is_null < 0.5) and r.addr_nums and r.first_street_word:
                digit = sorted(list(r.addr_nums))[0]
                k_addr = f'num_{digit}_{r.first_street_word}'
                lst_addr = self.index[k_addr]
                if len(lst_addr) < self.max_posting:
                    lst_addr.append((flag, i))

    def query(self, parsed_s1: ParsedRecord) -> List[Tuple[int, int]]:
        sw = self.stopwords
        df = self.df
        cand_scores = Counter()
        sig = parsed_s1.name_sig
        if len(sig) >= 8 and f'p8_{sig[:8]}' in self.index:
            for item in self.index[f'p8_{sig[:8]}']:
                cand_scores[item] += 3
        elif len(sig) >= 5 and f'p6_{sig[:6]}' in self.index:
            for item in self.index[f'p6_{sig[:6]}']:
                cand_scores[item] += 2
        toks = [w for w in parsed_s1.name_words_set if len(w) >= 3 and (w not in sw) and (2 <= df.get(w, 0) <= 300)]
        toks.sort(key=lambda w: df.get(w, 0))
        for w in toks[:3]:
            for item in self.index.get(f't_{w}', []):
                cand_scores[item] += 1
        if (parsed_s1.is_null < 0.5) and parsed_s1.addr_nums and parsed_s1.first_street_word:
            digit = sorted(list(parsed_s1.addr_nums))[0]
            for item in self.index.get(f'num_{digit}_{parsed_s1.first_street_word}', []):
                cand_scores[item] += 2
        return [c for c, _ in cand_scores.most_common(self.max_candidates)]

print('Adaptive Blocking Index Ready!')

In [ ]:
# 7. Load Full Dataset, Parse & Extract Training Pairs
SAMPLE_SIZE = 100000  # Set to 0 for full dataset (~350k entities), or 100k for fast training
RANDOM_STATE = 42

print(f'Loading data from {DATA_DIR}...')
s1_path = os.path.join(DATA_DIR, 'train_source1.tsv')
gt_path = os.path.join(DATA_DIR, 'train_ground_truth.tsv')
s2_path = os.path.join(DATA_DIR, 'train_source2.tsv')
s3_path = os.path.join(DATA_DIR, 'train_source3.tsv')

if SAMPLE_SIZE > 0:
    s1_df = pl.read_csv(s1_path, separator='\t', n_rows=SAMPLE_SIZE)
else:
    s1_df = pl.read_csv(s1_path, separator='\t')

gt_df = pl.read_csv(gt_path, separator='\t')
s2_df = pl.read_csv(s2_path, separator='\t')
s3_df = pl.read_csv(s3_path, separator='\t')

s1_ids = s1_df['entity_id'].to_list()
s1_names = s1_df['business_name'].to_list()
s1_addrs = s1_df['business_address'].to_list()

s2_ids = s2_df['entity_id'].to_list()
s2_names = s2_df['business_name'].to_list()
s2_addrs = s2_df['business_address'].to_list()

s3_ids = s3_df['entity_id'].to_list()
s3_names = s3_df['business_name'].to_list()
s3_addrs = s3_df['business_address'].to_list()

print(f'Parsing records: S1={len(s1_ids):,}, S2={len(s2_ids):,}, S3={len(s3_ids):,}...')
s1_parsed = [parse_record(n, a) for n, a in zip(s1_names, s1_addrs)]
s2_parsed = [parse_record(n, a) for n, a in zip(s2_names, s2_addrs)]
s3_parsed = [parse_record(n, a) for n, a in zip(s3_names, s3_addrs)]

# Build Ground Truth Map
gt_map: Dict[str, Set[str]] = {sid: set() for sid in s1_ids}
for row in gt_df.iter_rows(named=True):
    sid = row['source1_entity_id']
    if sid in gt_map:
        raw_m = row.get('matched_entity_ids')
        if raw_m is not None and str(raw_m) != 'None' and str(raw_m) != 'nan' and str(raw_m).strip():
            gt_map[sid] = set(str(raw_m).strip().split(','))

# Compute Dynamic Corpus IDF
corpus = compute_corpus_idf(s2_parsed + s3_parsed)
print(f'Corpus DF size: {len(corpus.df):,}, Stopwords identified: {len(corpus.stopwords):,}')

# Build Inverted Index
index = AdaptiveInvertedIndex(max_posting=30, stopwords=corpus.stopwords, df_dict=corpus.df)
index.add_records(s2_parsed, is_s2_flag=1)
index.add_records(s3_parsed, is_s2_flag=0)
print('Index populated successfully!')

In [ ]:
# 8. Train/Val Split & Measure Blocking Recall Ceiling (TASK 2) & Hard Negative Sampling (TASK 3)
s1_indices = np.arange(len(s1_ids))
is_singleton = np.array([1 if len(gt_map[sid]) == 0 else 0 for sid in s1_ids])

train_idx, val_idx = train_test_split(
    s1_indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=is_singleton
)

train_s1_ids = set(s1_ids[i] for i in train_idx)
val_s1_ids = set(s1_ids[i] for i in val_idx)
val_gt_dict = {sid: gt_map[sid] for sid in val_s1_ids}

s2_map = {eid: p for eid, p in zip(s2_ids, s2_parsed)}
s3_map = {eid: p for eid, p in zip(s3_ids, s3_parsed)}

# Measure Validation Blocking Recall Ceiling
print('=' * 70)
print('TASK 2: MEASURING BLOCKING RECALL CEILING ON HELD-OUT SPLIT')
print('=' * 70)
val_candidates_dict = {}
val_cand_pairs = []

for i in val_idx:
    sid = s1_ids[i]
    p1 = s1_parsed[i]
    cands = index.query(p1)
    cand_ids = set()
    for is_s2, row_idx in cands:
        tid = s2_ids[row_idx] if is_s2 == 1 else s3_ids[row_idx]
        cand_ids.add(tid)
        p2 = s2_parsed[row_idx] if is_s2 == 1 else s3_parsed[row_idx]
        feats = compute_parsed_features(p1, p2, is_s2 == 1, corpus.idf, corpus.default_idf)
        val_cand_pairs.append((sid, tid, feats))
    val_candidates_dict[sid] = cand_ids

blocking_res = evaluate_blocking_recall(
    val_gt_dict,
    val_candidates_dict,
    total_s2_pool_size=len(s2_ids),
    total_s3_pool_size=len(s3_ids)
)

print(f'-> Blocking Recall Ceiling : {blocking_res["blocking_recall_ceiling"]*100:.2f}%')
print(f'-> Retained True Matches   : {blocking_res["retained_in_candidates"]:,} / {blocking_res["total_true_matches"]:,}')
print(f'-> Missed in Blocking      : {blocking_res["missed_in_blocking"]:,}')
print(f'-> Full Entity Coverage    : {blocking_res["entity_full_coverage_rate"]*100:.2f}%')
print(f'-> Avg Candidates per S1   : {blocking_res["avg_candidates_per_s1"]:.2f}')
if 'reduction_ratio' in blocking_res:
    print(f'-> Reduction Ratio         : {blocking_res["reduction_ratio"]*100:.5f}%')
print('=' * 70)

# TASK 3: Mining Hard Negatives for Training Set
print('\nGenerating balanced training pairs with Hard Negatives...')
X_train, y_train = [], []
pos_train, neg_train = 0, 0

for i in train_idx:
    sid = s1_ids[i]
    p1 = s1_parsed[i]
    true_targets = gt_map.get(sid, set())
    for tid in true_targets:
        if tid in s2_map:
            feats = compute_parsed_features(p1, s2_map[tid], True, corpus.idf, corpus.default_idf)
            X_train.append(feats)
            y_train.append(1)
            pos_train += 1
        elif tid in s3_map:
            feats = compute_parsed_features(p1, s3_map[tid], False, corpus.idf, corpus.default_idf)
            X_train.append(feats)
            y_train.append(1)
            pos_train += 1

    cands = index.query(p1)
    for is_s2, row_idx in cands:
        tid = s2_ids[row_idx] if is_s2 == 1 else s3_ids[row_idx]
        if tid not in true_targets:
            p2 = s2_parsed[row_idx] if is_s2 == 1 else s3_parsed[row_idx]
            feats = compute_parsed_features(p1, p2, is_s2 == 1, corpus.idf, corpus.default_idf)
            X_train.append(feats)
            y_train.append(0)
            neg_train += 1

print(f'Training Instances: Total={len(X_train):,}, Positives={pos_train:,}, Hard Negatives={neg_train:,}')
X_train_arr = np.array(X_train, dtype=np.float32)
y_train_arr = np.array(y_train, dtype=np.int32)

X_val, y_val = [], []
for sid, tid, feats in val_cand_pairs:
    X_val.append(feats)
    y_val.append(1 if tid in val_gt_dict[sid] else 0)
X_val_arr = np.array(X_val, dtype=np.float32)
y_val_arr = np.array(y_val, dtype=np.int32)

In [ ]:
# 9. Train LightGBM with Monotonic Constraints
MONOTONE_CONSTRAINTS = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, -1, 1, 0, 0, 0]

print('Training LightGBM Matcher with Monotonic Constraints...')
model = lgb.LGBMClassifier(
    objective='binary',
    metric='binary_logloss',
    n_estimators=300,
    learning_rate=0.08,
    num_leaves=31,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    monotone_constraints=MONOTONE_CONSTRAINTS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

t0 = time.time()
model.fit(X_train_arr, y_train_arr, eval_set=[(X_val_arr, y_val_arr)], callbacks=[lgb.early_stopping(30, verbose=False)])
print(f'Model trained in {time.time() - t0:.2f}s! Best iteration: {model.best_iteration_}')

val_probs = model.predict_proba(X_val_arr)[:, 1]
val_auc = roc_auc_score(y_val_arr, val_probs)
print(f'Validation Pairwise ROC-AUC: {val_auc:.5f}')

# Feature Importances
for fname, imp in sorted(zip(FEATURE_NAMES, model.feature_importances_), key=lambda x: x[1], reverse=True):
    print(f'{fname:28s} : {imp:5d}')

In [ ]:
# 10. TASK 1: Exact Macro F_0.5 Threshold Optimization
print('=' * 70)
print('TASK 1: MACRO F0.5 THRESHOLD OPTIMIZATION (GRID SEARCH)')
print('=' * 70)

val_X_pairs = np.array([p[2] for p in val_cand_pairs], dtype=np.float32)
pair_probs = model.booster_.predict(val_X_pairs)

thresholds = np.linspace(0.40, 0.95, 29)
best_theta = 0.80
best_f05 = -1.0
best_report = {}

print(f'{"Threshold (theta)":18s} | {"Macro F0.5":12s} | {"Singleton Acc":14s} | {"Non-Sing F0.5":14s}')
print('-' * 65)

for theta in thresholds:
    best_assignment: Dict[str, Tuple[float, str]] = {}
    for (sid, cid, _), prob in zip(val_cand_pairs, pair_probs):
        if prob >= theta:
            if cid not in best_assignment or prob > best_assignment[cid][0]:
                best_assignment[cid] = (prob, sid)

    s1_preds: Dict[str, Set[str]] = {sid: set() for sid in val_s1_ids}
    for cid, (prob, sid) in best_assignment.items():
        s1_preds[sid].add(cid)

    report = detailed_evaluation_report(val_gt_dict, s1_preds)
    f05 = report['macro_f05']
    s_acc = report['singleton_accuracy']
    ns_f05 = report['non_singleton_macro_f05']

    print(f'theta = {theta:.2f}            | {f05:12.4f} | {s_acc:14.4f} | {ns_f05:14.4f}')

    if f05 > best_f05:
        best_f05 = f05
        best_theta = float(theta)
        best_report = report

print('-' * 65)
print(f'OPTIMAL THRESHOLD: theta* = {best_theta:.2f}')
print(f'  Overall Macro F0.5 : {best_f05:.4f}')
print(f'  Singleton Accuracy : {best_report.get("singleton_accuracy", 0.0)*100:.2f}%')
print(f'  Non-Singleton F0.5 : {best_report.get("non_singleton_macro_f05", 0.0):.4f}')
print(f'  Global Micro Prec  : {best_report.get("global_micro_precision", 0.0):.4f}')
print(f'  Global Micro Rec   : {best_report.get("global_micro_recall", 0.0):.4f}')

In [ ]:
# 11. CPU Prediction Throughput Benchmark & Artifact Export
print('Benchmarking CPU Throughput...')
rng = np.random.RandomState(42)
X_bench = rng.uniform(0.0, 1.0, size=(200000, len(FEATURE_NAMES))).astype(np.float32)
t0 = time.time()
_ = model.booster_.predict(X_bench)
t_el = time.time() - t0
speed = 200000 / t_el
print(f'CPU Throughput: {speed:,.0f} predictions/sec')

# Save model & metadata to /kaggle/working/ and code/business_entity_resolution/src/models/
out_dir = '/kaggle/working'
os.makedirs(out_dir, exist_ok=True)
model_file = os.path.join(out_dir, 'adaptive_matcher.pkl')
meta_file = os.path.join(out_dir, 'adaptive_matcher_meta.json')

with open(model_file, 'wb') as f:
    pickle.dump(model, f)

metadata = {
    'model_name': 'LightGBM Adaptive Business Matcher',
    'optimal_threshold': float(best_theta),
    'validation_macro_f05': float(best_f05),
    'validation_pairwise_auc': float(val_auc),
    'blocking_recall_ceiling': float(blocking_res['blocking_recall_ceiling']),
    'avg_candidates_per_s1': float(blocking_res['avg_candidates_per_s1']),
    'cpu_throughput_preds_per_sec': float(speed)
}

with open(meta_file, 'w') as f:
    json.dump(metadata, f, indent=2)

print('Trained artifacts successfully saved to /kaggle/working/!')